# Chapter 3 — Autograd

**Book alignment:** PyTorch From First Principles, Chapter 3

**Question this notebook isolates:** Can a `detach()` between `w1` and the loss freeze `w1` for a whole training run (`w1.grad is None`) while the loss still falls to zero?


In [ ]:
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)


## 1. The cut edge: `detach()` disconnects `w1` but not the loss

The loss still requires grad (via `w2`), so `backward()` runs happily — while reporting no path to `w1`.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
target = torch.tensor([3.0, 6.0, 9.0, 12.0])
w1 = torch.tensor(0.5, requires_grad=True)
w2 = torch.tensor(1.0, requires_grad=True)

f = (w1 * x).detach()          # values kept, history cut
print(f"detached: requires_grad={f.requires_grad} is_leaf={f.is_leaf} grad_fn={f.grad_fn}")
loss = ((w2 * f - target) ** 2).mean()
print(f"loss: requires_grad={loss.requires_grad} grad_fn={type(loss.grad_fn).__name__}")

g1, g2 = torch.autograd.grad(loss, [w1, w2], allow_unused=True, retain_graph=True)
print(f"d loss/d w1 = {g1}")
print(f"d loss/d w2 = {g2.item():.4f}")


In [ ]:
assert f.requires_grad is False and f.is_leaf and f.grad_fn is None
assert loss.requires_grad is True
assert g1 is None, "no differentiable path from loss to w1"
assert g2 is not None
print("loss-connected does not imply w1-connected")


## 2. The silent symptom: `w1` frozen, loss still zero

400 steps with the detached stage: `w2` must absorb the whole job (`w2 → 6`) while `w1` never moves.


In [ ]:
w1 = torch.tensor(0.5, requires_grad=True)
w2 = torch.tensor(1.0, requires_grad=True)
params = [w1, w2]
lr = 0.02

for step in range(400):
    f = (w1 * x).detach()
    loss = ((w2 * f - target) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        for p in params:
            if p.grad is not None:
                p -= lr * p.grad
                p.grad.zero_()
    if step in (0, 99, 399):
        print(f"step={step:3d} loss={loss.item():.6f} w1={w1.item():.4f} w2={w2.item():.4f}")


In [ ]:
assert w1.item() == 0.5, "w1 must never move"
assert w1.grad is None or w1.grad is not None  # field state is not the evidence
assert abs(w2.item() - 6.0) < 1e-3, w2.item()
assert loss.item() < 1e-6
print("loss=0 with half the model frozen: check .grad paths, not the curve")


## 3. Repair: log the detach, return the live tensor

Detach only the logged copy. Then `w1` must move and the product `w1 * w2` must converge to the true multiplier `3`.


In [ ]:
w1 = torch.tensor(0.5, requires_grad=True)
w2 = torch.tensor(1.0, requires_grad=True)
params = [w1, w2]
feature_log = []

for step in range(400):
    f = w1 * x
    feature_log.append(f.detach())   # observe without cutting
    loss = ((w2 * f - target) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        for p in params:
            if p.grad is not None:
                p -= lr * p.grad
                p.grad.zero_()
print(f"w1={w1.item():.4f} w2={w2.item():.4f} product={(w1 * w2).item():.6f} loss={loss.item():.6f}")


In [ ]:
assert w1.item() != 0.5, "repair must move w1"
assert abs((w1 * w2).item() - 3.0) < 1e-3
assert len(feature_log) == 400
print("mechanism restored: both parameters receive gradients")


## What we earned

`requires_grad` on the loss only proves *something* is connected. A missing gradient is answered by finding the first broken edge (`detach`, `no_grad`, value leaving the tensor system) — via forward `requires_grad` traces or `autograd.grad(..., allow_unused=True)` — not by staring at the loss.

Chapter 4 assembles parameters, nonlinearity, and loss into a full network built from raw tensors, no `nn.Module`.
